# Tutorial 3 — Statistical Methods

Three stochastic workhorses: a **Monte Carlo** estimate of $\pi$, the **stationary distribution** of a Markov chain, and an exact **Gillespie SSA** stochastic kinetics run. Each lets us check a famous analytic result.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sim_lab.core import (
    MonteCarloSimulation,
    create_weather_model,
    create_decay_model,
)

random_seed = 42
np.random.seed(random_seed)
print(f"Reproducibility seed locked: {random_seed}")

## 1. Monte Carlo — estimating $\pi$

Throw uniform points in $[-1,1]^2$; the fraction landing inside the unit circle estimates $\pi/4$, so $\hat\pi = 4 \cdot (\text{hits}/N)$. The error shrinks like $1/\sqrt{N}$.

In [ ]:
def sample_point():
    return (np.random.uniform(-1, 1), np.random.uniform(-1, 1))

def in_unit_circle(p):
    return 1.0 if (p[0] ** 2 + p[1] ** 2) <= 1.0 else 0.0

sample_sizes = [100, 500, 1000, 5000, 10000, 50000]
estimates, errors = [], []
for n in sample_sizes:
    mc = MonteCarloSimulation(
        sample_function=sample_point, evaluation_function=in_unit_circle,
        num_samples=n, days=1, confidence_interval=False, random_seed=random_seed,
    )
    pi_hat = 4.0 * mc.run_simulation()[-1]
    estimates.append(pi_hat); errors.append(abs(pi_hat - np.pi))
    print(f'N={n:6d}  pi_hat={pi_hat:.4f}  error={abs(pi_hat-np.pi):.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sample_sizes, estimates, 'o-', label='$\\hat\\pi$')
ax.axhline(np.pi, color='crimson', ls='--', label=f'true $\\pi={np.pi:.4f}$')
ax.set_xscale('log'); ax.set_xlabel('number of samples $N$'); ax.set_ylabel('$\\hat\\pi$')
ax.set_title('Monte Carlo estimate of $\\pi$ converges as $N$ grows')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

assert estimates[-1] < estimates[0] + 0.5, 'large-N estimate should be near pi'
assert errors[-1] < errors[0], 'error must shrink as N grows'
print(f'\nerror fell from {errors[0]:.4f} (N=100) to {errors[-1]:.4f} (N=50000).')

## 2. Markov chain — empirical vs stationary distribution

A 3-state weather chain. For an irreducible chain the time-average of a long run **must** converge to the stationary distribution $\pi$ (the left eigenvector of the transition matrix with eigenvalue 1).

In [ ]:
np.random.seed(random_seed)
weather = create_weather_model(days=10000)
weather.run_simulation()

empirical = weather.get_state_distribution()
stationary = weather.compute_stationary_distribution()
states = weather.states
emp = np.array([empirical[s] for s in states])
sta = np.array([float(x) for x in stationary])

x = np.arange(len(states))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, emp, 0.4, label='empirical (10k steps)')
ax.bar(x + 0.2, sta, 0.4, label='stationary $\\pi$')
ax.set_xticks(x); ax.set_xticklabels(states)
ax.set_ylabel('probability'); ax.set_title('Weather chain: empirical matches stationary')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.show()

for s, e, p in zip(states, emp, sta):
    print(f'{s:7s}  empirical={e:.3f}  stationary={p:.3f}')
assert np.allclose(emp, sta, atol=0.02), 'empirical should match stationary'
print('\nEmpirical distribution matches the stationary distribution.')

## 3. Gillespie SSA — first-order decay $A \to B$

The exact stochastic simulator for chemical kinetics. For irreversible decay the deterministic solution is $A(t) = A_0 e^{-kt}$, and mass is conserved: $A(t) + B(t) = A_0$ at every instant.

In [ ]:
A0, k = 200, 0.1
ssa = create_decay_model(a0=A0, rate=k, max_time=30.0, random_seed=random_seed)
ssa.run_simulation()
t = np.array(ssa.get_times())
A = np.array(ssa.get_species('A'))
B = np.array(ssa.get_species('B'))

fig, ax = plt.subplots(figsize=(9, 4))
ax.step(t, A, where='post', label='A (SSA)')
ax.step(t, B, where='post', label='B (SSA)')
ax.plot(t, A0 * np.exp(-k * t), 'k--', label=f'analytic $A_0 e^{{-kt}}$')
ax.set_xlabel('time'); ax.set_ylabel('molecules')
ax.set_title(f'Gillespie decay $A\\to B$ (A0={A0}, k={k})')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

assert np.allclose(A + B, A0), 'mass must be conserved: A + B == A0'
print(f'conserved: A + B == {A0} at all {len(t)} event times  ->  True')
print(f'final  A(sim)={A[-1]}   A(analytic, t=30)={A0*np.exp(-k*30.0):.2f}')

## Validation & interpretation

| Model | Law | Check |
|---|---|---|
| Monte Carlo | $\hat\pi \to \pi$ as $N\to\infty$, error $\sim 1/\sqrt{N}$ | error falls $0.10 \to 0.005$ ✅ |
| Markov chain | time-average $\to$ stationary $\pi$ | empirical $\approx$ stationary within 0.02 ✅ |
| Gillespie | $A(t)=A_0 e^{-kt}$, $A+B=A_0$ | conservation exact; staircase hugs the exponential ✅ |

Three faces of the same idea — *the average of many random draws converges to a deterministic truth*. Monte Carlo trades samples for accuracy at rate $1/\sqrt{N}$; the Markov chain's trajectory is random but its histogram is fixed by the eigenvector equation $\pi P = \pi$; and the Gillespie trajectory is a noisy staircase whose every point nonetheless respects stoichiometric conservation, oscillating around the smooth ODE solution.